# Translate the CSV's unique text strings (Hindi + Mandarin) on Colab GPU

Before running: **Runtime -> Change runtime type -> Hardware accelerator -> GPU** (T4 or whatever's available) -> Save.

Context: the merged VLM dataset has 157,514 rows, but `radlex_phrase` (the
text actually being translated for 99% of rows) is templated -- only 3,379
distinct strings appear across all 156,461 rows with anything to translate.
So instead of translating every row, this notebook translates just the
3,379 unique strings once with each fine-tuned checkpoint; a separate local
script (`merge_translations.py`, run outside Colab, no GPU needed) maps the
results back onto every matching row.

No GitHub access needed -- `translate_texts.py` has no local imports, so
this notebook writes it directly to the Colab VM instead of cloning the
(private) repo.

**Before running this notebook**, upload `data/to_translate/unique_texts.txt`
(produced locally by `python src/extract_unique_texts.py`) to
`My Drive/medical_vlm_data/to_translate/unique_texts.txt`. Both fine-tuned
checkpoints are expected already in Drive from the training notebooks:
`medical_vlm_checkpoints/opus-mt-en-hi-medical/final` and
`medical_vlm_checkpoints/opus-mt-en-zh-medical/final`.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- stop and pick a GPU runtime')
assert torch.cuda.is_available(), 'No GPU attached -- use Runtime -> Change runtime type -> GPU'

In [ ]:
# Deliberately NOT reinstalling torch -- Colab's preinstalled torch already has
# CUDA wired up correctly; reinstalling from requirements.txt risks silently
# swapping it for a CPU-only build.
!pip install -q transformers sentencepiece accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = "/content/repo"
os.makedirs(f"{REPO_DIR}/src", exist_ok=True)
os.makedirs(f"{REPO_DIR}/data/to_translate", exist_ok=True)
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
%%writefile src/translate_texts.py
"""Batch-translate a plain-text file (one sentence per line) with a fine-tuned
MarianMT checkpoint, preserving line order and count 1:1 with the input.

Built for the CSV translation task: `text_to_translate` in the merged VLM
dataset is 99%+ populated by a templated `radlex_phrase` field, so only
~3,379 distinct strings exist across 156,461 rows (see
`extract_unique_texts.py`) -- this script translates just those unique
strings once; mapping the results back onto every row is a separate,
model-free step (`merge_translations.py`).

Usage:
    python src/translate_texts.py \
        --model models/opus-mt-en-hi-medical/final \
        --in data/to_translate/unique_texts.txt \
        --out data/to_translate/unique_texts_hi.txt \
        --batch-size 32
"""
import argparse
from pathlib import Path

import torch
from transformers import MarianMTModel, MarianTokenizer


def detokenize_segmented_zh(text: str) -> str:
    return text.replace(" ", "").strip()


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True, help="HF hub ID or local checkpoint dir")
    ap.add_argument("--in", dest="in_path", required=True, type=Path)
    ap.add_argument("--out", dest="out_path", required=True, type=Path)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--max-length", type=int, default=128)
    ap.add_argument("--detok-zh", action="store_true",
                     help="strip the space-segmentation artifact this project's zh checkpoint was trained to emit (see docs/process_guide.md)")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Loading checkpoint: {args.model}")
    tokenizer = MarianTokenizer.from_pretrained(args.model)
    model = MarianMTModel.from_pretrained(args.model).to(device)
    model.eval()

    lines = args.in_path.read_text(encoding="utf-8").splitlines()
    print(f"Translating {len(lines)} lines, batch size {args.batch_size}")

    outputs = []
    for start in range(0, len(lines), args.batch_size):
        batch = lines[start:start + args.batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True,
                            max_length=args.max_length, padding=True).to(device)
        with torch.no_grad():
            generated = model.generate(**inputs, max_length=args.max_length)
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        if args.detok_zh:
            decoded = [detokenize_segmented_zh(d) for d in decoded]
        outputs.extend(decoded)
        done = min(start + args.batch_size, len(lines))
        print(f"  {done}/{len(lines)}", end="\r")

    print()
    assert len(outputs) == len(lines), f"line count mismatch: {len(outputs)} outputs vs {len(lines)} inputs"

    args.out_path.parent.mkdir(parents=True, exist_ok=True)
    args.out_path.write_text("\n".join(outputs) + "\n", encoding="utf-8")
    print(f"Wrote {args.out_path}")


if __name__ == "__main__":
    main()

In [ ]:
import os, shutil

os.chdir("/content/repo")  # hardcoded so this cell is self-contained even after a partial re-run

DRIVE_INPUT = "/content/drive/MyDrive/medical_vlm_data/to_translate/unique_texts.txt"
assert os.path.isfile(DRIVE_INPUT), f"{DRIVE_INPUT} not found -- upload data/to_translate/unique_texts.txt there first"
shutil.copy(DRIVE_INPUT, "data/to_translate/unique_texts.txt")

n_lines = sum(1 for _ in open("data/to_translate/unique_texts.txt", encoding="utf-8"))
print(f"Copied unique_texts.txt: {n_lines} lines")

## Translate into Hindi

Loads the fine-tuned Stage 2 checkpoint straight from Drive (a one-time load
for a single-pass job, so no need to copy ~300MB locally first).

In [ ]:
import os

os.chdir("/content/repo")

HI_MODEL = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-hi-medical/final"
assert os.path.isdir(HI_MODEL), f"{HI_MODEL} not found -- run colab_train_en_hi.ipynb first"

!python src/translate_texts.py --model "$HI_MODEL" --in data/to_translate/unique_texts.txt --out data/to_translate/unique_texts_hi.txt --batch-size 32

## Translate into Mandarin

`--detok-zh` strips the space-segmentation this checkpoint was trained to
emit (see `docs/process_guide.md` / the Mandarin tokenization bug note) --
without it the Chinese output would have a stray space between every
character.

In [ ]:
import os

os.chdir("/content/repo")

ZH_MODEL = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-zh-medical/final"
assert os.path.isdir(ZH_MODEL), f"{ZH_MODEL} not found -- run colab_train_en_zh.ipynb first"

!python src/translate_texts.py --model "$ZH_MODEL" --in data/to_translate/unique_texts.txt --out data/to_translate/unique_texts_zh.txt --batch-size 32 --detok-zh

In [ ]:
# Quick sanity check before trusting the output -- print a few aligned
# (en, hi, zh) triples so a human can eyeball them, same spirit as the
# training notebooks' post-run sanity checks.
import os
os.chdir("/content/repo")

en = open("data/to_translate/unique_texts.txt", encoding="utf-8").read().splitlines()
hi = open("data/to_translate/unique_texts_hi.txt", encoding="utf-8").read().splitlines()
zh = open("data/to_translate/unique_texts_zh.txt", encoding="utf-8").read().splitlines()

assert len(en) == len(hi) == len(zh), f"line count mismatch: en={len(en)} hi={len(hi)} zh={len(zh)}"
print(f"{len(en)} lines, all three files aligned.\n")

for i in [0, 1, len(en) // 2, len(en) - 1]:
    print(f"[{i}] EN: {en[i]}")
    print(f"     HI: {hi[i]}")
    print(f"     ZH: {zh[i]}")
    print()

In [ ]:
import os, shutil

os.chdir("/content/repo")

OUT_DIR = "/content/drive/MyDrive/medical_vlm_data/to_translate"
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy("data/to_translate/unique_texts_hi.txt", OUT_DIR)
shutil.copy("data/to_translate/unique_texts_zh.txt", OUT_DIR)
print(f"Copied unique_texts_hi.txt and unique_texts_zh.txt to {OUT_DIR}")
print("Download both from Drive, then run src/merge_translations.py locally (no GPU needed) to build the final CSV.")